In [ ]:
%pip install pandas mlxtend

In [ ]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

file_path = "Grocery_Dataset.csv"
df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

required_columns = {"TransactionID", "Item"}
if not required_columns.issubset(df.columns):
    raise ValueError(
        "The CSV must contain 'TransactionID' and 'Item'. "
        f"Found: {df.columns.tolist()}"
    )

df = df.dropna(subset=["TransactionID", "Item"]).copy()
df["Item"] = df["Item"].astype(str).str.strip()
df = df[df["Item"] != ""]

# Group purchased items by transaction; remove duplicate items within a transaction.
transactions = (
    df.groupby("TransactionID", sort=False)["Item"]
      .apply(lambda items: sorted(set(items)))
      .tolist()
)

print("Total transactions:", len(transactions))
print("Unique items:", len({item for transaction in transactions for item in transaction}))
print("First five transactions:")
for i, transaction in enumerate(transactions[:5], start=1):
    print(f"T{i}: {transaction}")


In [ ]:
encoder = TransactionEncoder()
encoded = encoder.fit(transactions).transform(transactions)

basket = pd.DataFrame(encoded, columns=encoder.columns_)

print("Transaction matrix shape:", basket.shape)
display(basket.head())


In [ ]:
MIN_SUPPORT = 0.40
MIN_CONFIDENCE = 0.60

frequent_itemsets = apriori(
    basket,
    min_support=MIN_SUPPORT,
    use_colnames=True
)

if frequent_itemsets.empty:
    print("No itemsets meet the minimum support threshold.")
else:
    # Keep the original frozenset values for association_rules().
    frequent_itemsets = frequent_itemsets.sort_values(
        ["support", "itemsets"], ascending=[False, True]
    ).reset_index(drop=True)

    display_items = frequent_itemsets.copy()
    display_items["itemsets"] = display_items["itemsets"].apply(
        lambda itemset: ", ".join(sorted(itemset))
    )
    display_items["support (%)"] = (display_items["support"] * 100).round(2)
    display_items = display_items.rename(columns={"itemsets": "Itemset"})
    display(display_items[["Itemset", "support", "support (%)"]])

print("Number of frequent itemsets:", len(frequent_itemsets))


In [ ]:
if frequent_itemsets.empty:
    rules = pd.DataFrame()
else:
    rules = association_rules(
        frequent_itemsets,
        metric="confidence",
        min_threshold=MIN_CONFIDENCE
    )

if rules.empty:
    print("No association rules meet the minimum confidence threshold.")
else:
    # Make readable copies for display; keep `rules`' original itemsets unchanged.
    display_rules = rules.copy()
    display_rules["antecedents"] = display_rules["antecedents"].apply(
        lambda items: ", ".join(sorted(items))
    )
    display_rules["consequents"] = display_rules["consequents"].apply(
        lambda items: ", ".join(sorted(items))
    )
    display_rules["support (%)"] = (display_rules["support"] * 100).round(2)
    display_rules["confidence (%)"] = (display_rules["confidence"] * 100).round(2)

    display_rules = display_rules.sort_values(
        ["confidence", "support"], ascending=False
    )
    display(display_rules[
        ["antecedents", "consequents", "support (%)", "confidence (%)"]
    ].rename(columns={
        "antecedents": "Antecedent (X)",
        "consequents": "Consequent (Y)"
    }))

print("Number of association rules:", len(rules))


In [ ]:
frequent_itemsets_export = frequent_itemsets.copy()
if not frequent_itemsets_export.empty:
    frequent_itemsets_export["itemsets"] = frequent_itemsets_export["itemsets"].apply(
        lambda items: ", ".join(sorted(items))
    )
frequent_itemsets_export.to_csv("grocery_frequent_itemsets_library.csv", index=False)

rules_export = rules.copy()
if not rules_export.empty:
    rules_export["antecedents"] = rules_export["antecedents"].apply(
        lambda items: ", ".join(sorted(items))
    )
    rules_export["consequents"] = rules_export["consequents"].apply(
        lambda items: ", ".join(sorted(items))
    )
rules_export.to_csv("grocery_association_rules_library.csv", index=False)

print("Saved: grocery_frequent_itemsets_library.csv")
print("Saved: grocery_association_rules_library.csv")
